In [1]:
!pip install pycryptodome boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 4.7 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [boto3]32m3/5 [s3transfer]


In [2]:
import sys
import logging
import base64
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
import hashlib
import hmac
import boto3
import csv
import os
import json

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("encryption.log"), logging.StreamHandler()],
)


def get_secrets():
    client = boto3.client(
        'secretsmanager',
        endpoint_url="",
        region_name="",
        aws_access_key_id="",
        aws_secret_access_key="",
        verify=False,
    )

    try:
        secret_name = ''
        response = client.get_secret_value(SecretId=secret_name)

        if 'SecretString' in response:
            secret = response['SecretString']

            try:
                secret_dict = json.loads(secret)
                aes_key = secret_dict['aes_key']
                hmac_key = secret_dict['hmac_key']
                iv = secret_dict['iv']

                return {'aes_key': aes_key, 'hmac_key': hmac_key, 'iv': iv}
            except json.JSONDecodeError as e:
                logging.error(f"JSONDecodeError: {e} - invalid JSON in secret {secret_name}. Secret content: {secret}")
                return None
        else:
            secret_binary = response['SecretBinary']
            return secret_binary

    except Exception as e:
        logging.error(f"Error retrieving secret: {e}")
        return None


def apply_aes_cbc_encryption(input_str):
    try:
        #encyption_keys = get_secrets()
        AES_KEY = 'YzglZ9JpxriJc6tFs2NFsWmLzxC4hz4kGyF4yNNHlQQ='
        HMAC_KEY = 'Z6iIilxZIHop8vZmS1CB6AwnmvSQvXlmqLm47bcwaFg=='
        IV_KEY = 'MTIzNDU2Nzg5MGFiY2RlZg=='

        aes_key = base64.b64decode(AES_KEY)
        hmac_key = base64.b64decode(HMAC_KEY)
        iv = base64.b64decode(IV_KEY)

        if not all([aes_key, hmac_key, iv]):
            raise ValueError("Missing required encryption secrets (aes_key, hmac_key, or iv).")

        if input_str is None:
            return None

        cipher = AES.new(aes_key, AES.MODE_CBC, iv)

        padded_pid = pad(input_str.encode('utf-8'), AES.block_size)

        ciphertext = cipher.encrypt(padded_pid)

        hmac_digest = hmac.new(hmac_key, iv + ciphertext, hashlib.sha256).digest()

        encrypted_data = base64.b64encode(iv + ciphertext + hmac_digest).decode('utf-8')

        return encrypted_data

    except Exception as e:
        logging.error(f"Encryption error: {e}")
        return input_str


def apply_aes_cbc_decryption(input_encrypted):
    # decryption_keys = get_secrets()
    # AES_KEY = decryption_keys.get('aes_key')
    # HMAC_KEY = decryption_keys.get('hmac_key')
    AES_KEY = 'YzglZ9JpxriJc6tFs2NFsWmLzxC4hz4kGyF4yNNHlQQ='
    HMAC_KEY = 'Z6iIilxZIHop8vZmS1CB6AwnmvSQvXlmqLm47bcwaFg=='
    IV_KEY = 'MTIzNDU2Nzg5MGFiY2RlZg=='

    aes_key = base64.b64decode(AES_KEY)
    hmac_key = base64.b64decode(HMAC_KEY)
    try:
        decoded_data = base64.b64decode(input_encrypted)

        # Extract IV, ciphertext, and HMAC
        iv = decoded_data[:16]
        ciphertext = decoded_data[16:-32]
        received_hmac = decoded_data[-32:]

        expected_hmac = hmac.new(hmac_key, iv + ciphertext, hashlib.sha256).digest()
        if not hmac.compare_digest(received_hmac, expected_hmac):
            raise ValueError("HMAC verification failed.")

        cipher = AES.new(aes_key, AES.MODE_CBC, iv=iv)
        padded_pid = cipher.decrypt(ciphertext)

        patient_id = unpad(padded_pid, AES.block_size).decode('utf-8')

        return patient_id

    except Exception as e:
        logging.error(f"Decryption error: {e}")
        return None


def main():
    # if len(sys.argv) != 3:
    #     logging.info("Usage: python script.py <encrypt/decrypt> <patient_id/encrypted_data>")
    #     sys.exit(1)
    #
    # operation = sys.argv[1].lower()
    # patient_id = sys.argv[2]
    #
    # try:
    #     if operation == "encrypt":
    #         encrypted_id = apply_aes_cbc_encryption(patient_id)
    #         logging.info(f"Encrypted patient ID: {encrypted_id}")
    #     elif operation == "decrypt":
    #         decrypted_id = apply_aes_cbc_decryption(patient_id)
    #         logging.info(f"Decrypted patient ID: {decrypted_id}")
    #     else:
    #         logging.info("Invalid operation. Use 'encrypt' or 'decrypt'.")
    #         sys.exit(1)
    # except Exception as e:
    #     logging.error(f"Error: {e}")
    #     sys.exit(1)
    plist = [
    "MTIzNDU2Nzg5MGFiY2RlZt80UeZSZkS7rKcRdcP68mH9A1Pd4aooauA5w6J1uaGdmKpbC9tyvelPBvzSOrtTIalnQsUFWzNwhlUEnUTussAxXoqQ4QoIWy2rXH/r4PvQ"
    ]


    with open('decrypted_ids.csv', mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Item', 'Decrypted ID'])  #Write the header

        for patient_id in plist:
            decrypted_id = apply_aes_cbc_decryption(patient_id)
            print(decrypted_id)
            #writer.writerow([patient_id, decrypted_id])  #Write the decrypted ID to the CSV file


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'Crypto'